In [1]:
%pip install -q langchain langchain-core langchain-google-genai langchain-classic langchain-community faiss-cpu python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
from dotenv import load_dotenv

from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import LLMChain, SimpleSequentialChain, SequentialChain, RetrievalQA
from langchain_classic.memory import (
    ConversationBufferMemory,
    ConversationBufferWindowMemory,
    ConversationSummaryMemory,
)
from langchain_community.vectorstores import FAISS
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

In [2]:
# ── Load API key ──────────────────────────────────────────────────────────
load_dotenv(override=True) 
gemini_api_key = os.getenv("GEMINI_API_KEY")
if not gemini_api_key:
    raise ValueError("GEMINI_API_KEY not found in .env file. Add it before continuing.")

# ── Initialise the Gemini LLM ────────────────────────────────────────
gemini_llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.3,
    max_output_tokens=512,
    google_api_key=gemini_api_key,
)

print("✅ All imports successful!")
print(f"   LLM  : {type(gemini_llm).__name__}")
print(f"   Model: gemini-2.5-flash")

# Quick sanity check
test = gemini_llm.invoke("What is the capital of France?")
print(f"\n🔍 Sanity check → {test.content.strip()}")

✅ All imports successful!
   LLM  : ChatGoogleGenerativeAI
   Model: gemini-2.5-flash

🔍 Sanity check → The capital of France is **Paris**.


In [3]:
print("🧠 SECTION 2: ConversationBufferMemory with Gemini")
print("=" * 55)

# ── Create memory object ──────────────────────────────────────────────────
memory = ConversationBufferMemory(memory_key="history", return_messages=False)

# ── Prompt template that includes {history} placeholder ───────────────────
conv_prompt = PromptTemplate(
    input_variables=["history", "input"],
    template=(
        "You are a friendly AI assistant. Use the conversation history below to answer.\n\n"
        "Conversation History:\n{history}\n"
        "Human: {input}\n"
        "AI:"
    ),
)

# ── LLMChain with memory ──────────────────────────────────────────────────
conversation_chain = LLMChain(
    llm=gemini_llm,
    prompt=conv_prompt,
    memory=memory,
    verbose=False,
)

# ── Simulate 3-turn conversation ──────────────────────────────────────────
turns = [
    "Hi! My name is Alice and I am a data scientist.",
    "I work mainly with Python and machine learning.",
    "I like to play guitar and sing.",
    "What is my name and what do I do in my free time?",   # tests memory recall
]

for i, user_input in enumerate(turns, 1):
    response = conversation_chain.predict(input=user_input)
    print(f"Turn {i}")
    print(f"  🧑 Human : {user_input}")
    print(f"  🤖 AI    : {response.strip()}")
    print()

# ── Inspect memory buffer ─────────────────────────────────────────────────
print("📋 Full Memory Buffer:")
print("-" * 40)
print(memory.buffer)

🧠 SECTION 2: ConversationBufferMemory with Gemini


C:\Users\Roni\AppData\Local\Temp\ipykernel_11104\85690259.py:5: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(memory_key="history", return_messages=False)
C:\Users\Roni\AppData\Local\Temp\ipykernel_11104\85690259.py:19: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  conversation_chain = LLMChain(


Turn 1
  🧑 Human : Hi! My name is Alice and I am a data scientist.
  🤖 AI    : Hi Alice! It's great to meet you. A data scientist, how interesting!

How can I help you today?

Turn 2
  🧑 Human : I work mainly with Python and machine learning.
  🤖 AI    : Ah, Python and machine learning! That's a very powerful combination, especially for a data scientist. How can I help you with that today?

Turn 3
  🧑 Human : I like to play guitar and sing.
  🤖 AI    : That's wonderful, Alice! A data scientist who also enjoys playing guitar and singing – what a lovely blend of analytical and creative talents!

Is there anything I can help you with regarding your music, or perhaps something else entirely?

Turn 4
  🧑 Human : What is my name and what do I do in my free time?
  🤖 AI    : Your name is Alice, and in your free time, you enjoy playing guitar and singing!

📋 Full Memory Buffer:
----------------------------------------
Human: Hi! My name is Alice and I am a data scientist.
AI: Hi Alice! It's gr

In [4]:
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

print("💬 SECTION 3: Conversation History with Gemini")
print("=" * 55)

# ── Build conversation history manually ───────────────────────────────────
chat_history = [
    SystemMessage(content="You are a knowledgeable AI assistant who remembers the entire conversation."),
    HumanMessage(content="Hello! My name is Bob and I'm a software engineer."),
    AIMessage(content="Hello Bob! Great to meet you. How can I help you today?"),
    HumanMessage(content="I'm learning about machine learning and neural networks."),
    AIMessage(content="That's fascinating! Machine learning is a great field to explore."),
]

# ── Ask a follow-up that requires history to answer correctly ─────────────
follow_up = HumanMessage(content="Based on what I told you, what is my name and what am I learning?")
chat_history.append(follow_up)

response = gemini_llm.invoke(chat_history)
print("📜 Chat History (before response):")
for msg in chat_history[:-1]:        # print everything except the follow-up
    role = type(msg).__name__.replace("Message", "")
    print(f"  [{role:6}] {msg.content}")

print(f"\n  [Human ] {follow_up.content}")
print(f"\n🤖 Gemini Response:\n   {response.content.strip()}")

# ── Add AI reply to history and continue ──────────────────────────────────
chat_history.append(AIMessage(content=response.content))
chat_history.append(HumanMessage(content="Recommend one book for me to read."))
final_resp = gemini_llm.invoke(chat_history)
print(f"\n🧑 Human : Recommend one book for me to read.")
print(f"🤖 AI    : {final_resp.content.strip()}")

💬 SECTION 3: Conversation History with Gemini
📜 Chat History (before response):
  [System] You are a knowledgeable AI assistant who remembers the entire conversation.
  [Human ] Hello! My name is Bob and I'm a software engineer.
  [AI    ] Hello Bob! Great to meet you. How can I help you today?
  [Human ] I'm learning about machine learning and neural networks.
  [AI    ] That's fascinating! Machine learning is a great field to explore.

  [Human ] Based on what I told you, what is my name and what am I learning?

🤖 Gemini Response:
   Your name is **Bob**, and you are learning about **machine learning and neural networks**.

🧑 Human : Recommend one book for me to read.
🤖 AI    : Given your background as a software engineer and your interest in machine learning and neural networks, I recommend:


In [5]:
print("🪟 SECTION 4: ConversationBufferWindowMemory (k=2) with Gemini")
print("=" * 60)

# ── Window memory – keeps only last k=2 exchanges ─────────────────────────
window_memory = ConversationBufferWindowMemory(
    k=2,                     # keep only last 2 human+AI pairs
    memory_key="history",
    return_messages=False,
)

window_prompt = PromptTemplate(
    input_variables=["history", "input"],
    template=(
        "You are a helpful AI assistant. Here is the recent conversation:\n\n"
        "{history}\n"
        "Human: {input}\n"
        "AI:"
    ),
)

window_chain = LLMChain(
    llm=gemini_llm,
    prompt=window_prompt,
    memory=window_memory,
    verbose=False,
)

# ── Run 5 turns – only last 2 will be in memory ───────────────────────────
messages = [
    "My favourite colour is blue.",
    "I enjoy playing chess.",
    "I live in Chennai.",
    "My hobby is photography.",
    "What do you remember about me?",   # only last 2 turns survive in window
]

for i, msg in enumerate(messages, 1):
    resp = window_chain.predict(input=msg)
    print(f"Turn {i}: 🧑 {msg}")
    print(f"         🤖 {resp.strip()[:120]}")
    print(f"         [Memory size: {len(window_memory.buffer.strip())} chars]")
    print()

🪟 SECTION 4: ConversationBufferWindowMemory (k=2) with Gemini


C:\Users\Roni\AppData\Local\Temp\ipykernel_11104\3733290624.py:5: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  window_memory = ConversationBufferWindowMemory(


Turn 1: 🧑 My favourite colour is blue.
         🤖 Blue is a wonderful color! It's very popular and often associated with calmness and the sky or ocean.
         [Memory size: 141 chars]

Turn 2: 🧑 I enjoy playing chess.
         🤖 Chess is a fantastic game! It's a classic strategy game known for developing critical thinking and problem-solving skill
         [Memory size: 380 chars]

Turn 3: 🧑 I live in Chennai.
         🤖 Chennai is a vibrant and historic city! It's known for its rich culture, beautiful temples, delicious South Indian cuisi
         [Memory size: 617 chars]

Turn 4: 🧑 My hobby is photography.
         🤖 Photography is a wonderful hobby! Chennai, with its vibrant culture, historic architecture, beautiful beaches, and bustl
         [Memory size: 727 chars]

Turn 5: 🧑 What do you remember about me?
         🤖 You live in Chennai and your hobby is photography!
         [Memory size: 441 chars]



In [6]:
print("📝 SECTION 5: ConversationSummaryMemory with Gemini")
print("=" * 55)

# ── Summary memory – LLM summarises the conversation ─────────────────────
summary_memory = ConversationSummaryMemory(
    llm=gemini_llm,          # uses Gemini itself to write the summary
    memory_key="history",
    return_messages=False,
)

summary_prompt = PromptTemplate(
    input_variables=["history", "input"],
    template=(
        "You are a helpful assistant. Here is a summary of our conversation so far:\n\n"
        "{history}\n\n"
        "Human: {input}\n"
        "AI:"
    ),
)

summary_chain = LLMChain(
    llm=gemini_llm,
    prompt=summary_prompt,
    memory=summary_memory,
    verbose=False,
)

# ── Run several turns ─────────────────────────────────────────────────────
turns = [
    "Hi! I'm Carol, a high-school mathematics teacher.",
    "I'm preparing a lesson on quadratic equations for Grade 10.",
    "The students are struggling with factoring.",
    "What is a good real-world example to explain factoring?",
]

for i, msg in enumerate(turns, 1):
    resp = summary_chain.predict(input=msg)
    print(f"Turn {i}: 🧑 {msg}")
    print(f"         🤖 {resp.strip()[:150]}")
    print()

# ── Show the running summary ──────────────────────────────────────────────
print("📋 Current Summary Memory:")
print("-" * 40)
print(summary_memory.buffer)

📝 SECTION 5: ConversationSummaryMemory with Gemini


C:\Users\Roni\AppData\Local\Temp\ipykernel_11104\3242208871.py:5: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  summary_memory = ConversationSummaryMemory(


Turn 1: 🧑 Hi! I'm Carol, a high-school mathematics teacher.
         🤖 Hello Carol! It's a pleasure to meet you. A high-school mathematics teacher – that's a very important job!

How can I help you today?

Turn 2: 🧑 I'm preparing a lesson on quadratic equations for Grade 10.
         🤖 Excellent! Quadratic equations for Grade 10 is a crucial topic. I can help you brainstorm lesson ideas, create engaging examples, develop practice pro

Turn 3: 🧑 The students are struggling with factoring.
         🤖 I understand. Factoring can definitely be a challenging area for many students, but it's a crucial



ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 18.42608404s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTi

In [8]:
print("🔗 SECTION 6A: SimpleSequentialChain with Gemini")
print("=" * 55)
# Pipeline:  topic → [Chain1: outline] → [Chain2: blog post]

# ── Chain 1: Generate a topic outline ────────────────────────────────────
outline_prompt = PromptTemplate(
    input_variables=["topic"],
    template="Write a concise 3-point outline for a blog post about: {topic}. Just the outline, no content.",
)
outline_chain = LLMChain(llm=gemini_llm, prompt=outline_prompt)

# ── Chain 2: Expand outline into intro paragraph ──────────────────────────
intro_prompt = PromptTemplate(
    input_variables=["outline"],
    template="Using this outline:\n{outline}\n\nWrite a compelling 2-sentence blog introduction.",
)
intro_chain = LLMChain(llm=gemini_llm, prompt=intro_prompt)

# ── Combine into SimpleSequentialChain ───────────────────────────────────
simple_seq = SimpleSequentialChain(
    chains=[outline_chain, intro_chain],
    verbose=True,
)

topic = "the benefits of learning Python for data science"
print(f"📌 Topic: {topic}\n")
final_intro = simple_seq.run(topic)
print(f"\n📝 Final Blog Introduction:\n{final_intro.strip()}")

🔗 SECTION 6A: SimpleSequentialChain with Gemini
📌 Topic: the benefits of learning Python for data science



> Entering new SimpleSequentialChain chain...


C:\Users\Administrator\AppData\Local\Temp\2\ipykernel_2776\6184852.py:27: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 1.0. Use `invoke` instead.
  final_intro = simple_seq.run(topic)


Here's a concise 3-point outline:

1.  **Versatile Ecosystem & Broad Applicability:** Leverage a vast array of specialized libraries (NumPy, Pandas, Scikit-learn, etc.) and apply skills beyond just data analysis.
2.  **Beginner-Friendly & Productive Syntax:** Quickly grasp the language due to its readability, leading to faster development and iteration.
3.  **Strong Community Support & Industry Demand:** Benefit from extensive resources, continuous development, and high employability in the data science field.
Thinking about a career in data science or looking to enhance your analytical toolkit? Python has solidified its position as the undisputed champion, offering an unparalleled blend of power, accessibility, and career opportunities. Its unique combination of a vast

> Finished chain.

📝 Final Blog Introduction:
Thinking about a career in data science or looking to enhance your analytical toolkit? Python has solidified its position as the undisputed champion, offering an unparallel

In [10]:
print("🔗 SECTION 7B: SequentialChain (multiple named I/O) with Gemini")
print("=" * 60)
# Pipeline:  company_type → [Chain1: name] → [Chain2: tagline] → [Chain3: tweet]

# ── Chain 1: Company name ─────────────────────────────────────────────────
name_prompt = PromptTemplate(
    input_variables=["company_type"],
    template="Create a single creative startup name for a {company_type} company. Only the name.",
)
name_chain = LLMChain(llm=gemini_llm, prompt=name_prompt, output_key="company_name")

# ── Chain 2: Tagline ──────────────────────────────────────────────────────
tagline_prompt = PromptTemplate(
    input_variables=["company_name"],
    template="Write a catchy one-line tagline for the company named: {company_name}",
)
tagline_chain = LLMChain(llm=gemini_llm, prompt=tagline_prompt, output_key="tagline")

# ── Chain 3: Launch tweet ─────────────────────────────────────────────────
tweet_prompt = PromptTemplate(
    input_variables=["company_name", "tagline"],
    template=(
        "Write a Twitter/X launch announcement tweet (max 280 chars) for a company called '{company_name}' "
        "with tagline: '{tagline}'. Include relevant emojis."
    ),
)
tweet_chain = LLMChain(llm=gemini_llm, prompt=tweet_prompt, output_key="tweet")

# ── Combine into SequentialChain ──────────────────────────────────────────
full_chain = SequentialChain(
    chains=[name_chain, tagline_chain, tweet_chain],
    input_variables=["company_type"],                                     # chain input
    output_variables=["company_name", "tagline", "tweet"],                # expose all outputs
    verbose=True,
)

result = full_chain({"company_type": "AI-powered healthcare"})
print("\n📊 Results:")
print(f"  🏢 Company Name : {result['company_name'].strip()}")
print(f"  💡 Tagline      : {result['tagline'].strip()}")
print(f"  🐦 Tweet        : {result['tweet'].strip()}")

🔗 SECTION 7B: SequentialChain (multiple named I/O) with Gemini


> Entering new SequentialChain chain...


C:\Users\Administrator\AppData\Local\Temp\2\ipykernel_2776\2643308120.py:37: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 1.0. Use `invoke` instead.
  result = full_chain({"company_type": "AI-powered healthcare"})



> Finished chain.

📊 Results:
  🏢 Company Name : **AuraMed**
  💡 Tagline      : Here are a few catchy one-line taglines for AuraMed, playing on different aspects of "A
  🐦 Tweet        : Here are a few catchy one-line taglines for AuraMed, playing on different aspects of '
